In [1]:
!pip install torch torchvision

In [2]:
#Tạo tệp ánh xạ nhãn chung (class_to_idx.json)
import os
import json
import pandas as pd
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebook" or os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# 1. Tạo class_to_idx.json chung
train_df = pd.read_csv("data/splits/train.csv")
classes = sorted(train_df['class_name'].unique().tolist())
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

with open(models_dir / "class_to_idx.json", "w", encoding="utf-8") as f:
    json.dump(class_to_idx, f, ensure_ascii=False, indent=4)

print(f"Đã lưu ánh xạ nhãn chung tại models/class_to_idx.json: {class_to_idx}")

Đã lưu ánh xạ nhãn chung tại models/class_to_idx.json: {'Bacterial Gill Diseases': 0, 'Bacterial Red Diseases': 1, 'Epizootic Ulcerative Syndrome (EUS)': 2, 'Healthy Fish': 3}


In [3]:
#Định nghĩa FishDataset dùng chung class_to_idx.json và tính Class Weights
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class FishDataset(Dataset):
    def __init__(self, csv_file, class_to_idx_path, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
        
        with open(class_to_idx_path, "r", encoding="utf-8") as f:
            self.class_to_idx = json.load(f)
            
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        label_name = row['class_name']
        
        image = Image.open(img_path).convert('RGB')
        label = self.class_to_idx[label_name]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Tính class weights cho CrossEntropyLoss (CHỈ TÍNH TRÊN TẬP TRAIN)
train_labels = [class_to_idx[name] for name in train_df['class_name']]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print(f"Class Weights tính trên tập Train: {class_weights}")

Class Weights tính trên tập Train: [2.01635514 0.87348178 1.78305785 0.5560567 ]


In [10]:
# Định nghĩa DataLoader, Model (ResNet50) và Vòng lặp huấn luyện chuẩn (B3)
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import f1_score

# Preprocessing & Augmentation (CHỈ AUGMENTATION TRÊN TRAIN)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

json_path = models_dir / "class_to_idx.json"
train_dataset = FishDataset("data/splits/train.csv", json_path, transform=train_transform)
val_dataset = FishDataset("data/splits/val.csv", json_path, transform=val_transform)
test_dataset = FishDataset("data/splits/test.csv", json_path, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

# Khởi tạo mô hình ResNet50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

model_name = "resnet50"
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
# Thay đổi lớp fully connected cuối cùng của ResNet50 ứng với số lớp bệnh
model.fc = nn.Linear(model.fc.in_features, len(class_to_idx))
model = model.to(device)

# Đưa class weights lên device và gán cho CrossEntropyLoss
class_weights_tensor = class_weights_tensor.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10  
best_val_f1 = 0.0
best_model_path = models_dir / f"{model_name}_best.pth"

print(f"\n--- BẮT ĐẦU HUẤN LUYỆN VÀ ĐÁNH GIÁ: {model_name.upper()} ---")

for epoch in range(num_epochs):
    # --- TRAINING ---
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
    epoch_train_loss = train_loss / len(train_dataset)
    epoch_train_acc = train_correct / train_total
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
            val_preds.extend(predicted.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_correct / val_total
    epoch_val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc*100:.2f}% - Val Macro-F1: {epoch_val_f1:.4f}")
    
    # Lưu checkpoint tốt nhất dựa trên Val Macro-F1
    if epoch_val_f1 > best_val_f1:
        best_val_f1 = epoch_val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"--> Lưu checkpoint tốt nhất tại epoch {epoch+1} với Val Macro-F1: {best_val_f1:.4f}")

print(f"\nHuấn luyện hoàn tất! Model tốt nhất được lưu tại: {best_model_path}")

Đang sử dụng thiết bị: cpu
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\USER/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:08<00:00, 11.6MB/s]



--- BẮT ĐẦU HUẤN LUYỆN VÀ ĐÁNH GIÁ: RESNET50 ---
Epoch [1/10] | Train Loss: 0.6282 - Train Acc: 78.33% | Val Loss: 1.6193 - Val Acc: 54.87% - Val Macro-F1: 0.5268
--> Lưu checkpoint tốt nhất tại epoch 1 với Val Macro-F1: 0.5268
Epoch [2/10] | Train Loss: 0.3876 - Train Acc: 87.14% | Val Loss: 0.9612 - Val Acc: 69.91% - Val Macro-F1: 0.6748
--> Lưu checkpoint tốt nhất tại epoch 2 với Val Macro-F1: 0.6748
Epoch [3/10] | Train Loss: 0.1661 - Train Acc: 94.79% | Val Loss: 1.3527 - Val Acc: 70.80% - Val Macro-F1: 0.7161
--> Lưu checkpoint tốt nhất tại epoch 3 với Val Macro-F1: 0.7161
Epoch [4/10] | Train Loss: 0.1668 - Train Acc: 96.18% | Val Loss: 4.3673 - Val Acc: 53.10% - Val Macro-F1: 0.4740
Epoch [5/10] | Train Loss: 0.3682 - Train Acc: 91.19% | Val Loss: 12.4372 - Val Acc: 27.43% - Val Macro-F1: 0.2090
Epoch [6/10] | Train Loss: 0.2506 - Train Acc: 92.24% | Val Loss: 0.9452 - Val Acc: 74.34% - Val Macro-F1: 0.7317
--> Lưu checkpoint tốt nhất tại epoch 6 với Val Macro-F1: 0.7317
Epoch

In [11]:
#Đánh giá Test đúng một lần với Model tốt nhất & Xuất kết quả ra CSV B4
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# Nạp lại trọng số tốt nhất để đánh giá Test
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds = []
test_targets = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        
        test_preds.extend(predicted.cpu().numpy())
        test_targets.extend(labels.numpy())

# Tính toán các chỉ số tổng quan
test_acc = accuracy_score(test_targets, test_preds)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(test_targets, test_preds, average='macro', zero_division=0)

print("=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===")
print(f"Accuracy:    {test_acc * 100:.2f}%")
print(f"Precision:   {test_precision * 100:.2f}%")
print(f"Recall:      {test_recall * 100:.2f}%")
print(f"Macro-F1:    {test_f1 * 100:.2f}%")

# Chi tiết từng lớp bệnh
class_names = [name for name, idx in sorted(class_to_idx.items(), key=lambda x: x[1])]
report_dict = classification_report(test_targets, test_preds, target_names=class_names, output_dict=True, zero_division=0)
print("\n--- CLASSIFICATION REPORT CHI TIẾT ---")
print(classification_report(test_targets, test_preds, target_names=class_names, zero_division=0))

# Confusion Matrix
conf_mat = confusion_matrix(test_targets, test_preds)
print("\n--- CONFUSION MATRIX ---")
print(conf_mat)

# Lưu kết quả ra CSV để đưa vào báo cáo đồ án
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)

# Tạo DataFrame lưu metric chi tiết từng lớp
df_report = pd.DataFrame(report_dict).transpose()
df_report.to_csv(reports_dir / f"{model_name}_test_report.csv")

# Lưu tổng hợp metrics chung
summary_results = {
    "Model": model_name,
    "Test Accuracy": test_acc,
    "Test Precision (Macro)": test_precision,
    "Test Recall (Macro)": test_recall,
    "Test Macro-F1": test_f1,
    "Best Val Macro-F1": best_val_f1
}
df_summary = pd.DataFrame([summary_results])
df_summary.to_csv(reports_dir / f"{model_name}_summary_metrics.csv", index=False)

print(f"\nĐã lưu toàn bộ kết quả báo cáo ra thư mục reports/ thành công!")

=== KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ===
Accuracy:    73.33%
Precision:   71.33%
Recall:      75.10%
Macro-F1:    71.54%

--- CLASSIFICATION REPORT CHI TIẾT ---
                                     precision    recall  f1-score   support

            Bacterial Gill Diseases       0.71      0.81      0.76        31
             Bacterial Red Diseases       0.79      0.49      0.61        83
Epizootic Ulcerative Syndrome (EUS)       0.48      0.74      0.58        35
                       Healthy Fish       0.87      0.96      0.91        76

                           accuracy                           0.73       225
                          macro avg       0.71      0.75      0.72       225
                       weighted avg       0.76      0.73      0.73       225


--- CONFUSION MATRIX ---
[[25  0  4  2]
 [ 9 41 24  9]
 [ 0  9 26  0]
 [ 1  2  0 73]]

Đã lưu toàn bộ kết quả báo cáo ra thư mục reports/ thành công!
